# depth-lens quick start

Train a tiny OpenMythos, then probe its accuracy on the K-hop modular composition task as you sweep `n_loops`. This is the smallest possible end-to-end demonstration of depth-lens.

Total time on RTX 4080 SUPER: about 3 minutes.

In [ ]:
from depth_lens import probe
from depth_lens.tasks import get_task
from depth_lens.adapters.openmythos_adapter import train_for_task, TrainConfig

task = get_task('k-hop')
print(task.description)

## 1. Generate a few task instances so you can see what the model is solving

In [ ]:
for inst in task.generate(depth=4, n_samples=3, seed=0):
    print(f'prompt: {inst.prompt!r}   target: {inst.target}')

## 2. Train a tiny OpenMythos (~2 min on a consumer GPU)

The bundled helper trains on a mix of K values so the model learns composition rather than memorising one hop count.

In [ ]:
adapter = train_for_task(task, cfg=TrainConfig(steps=2500))

## 3. Probe it across (depth × n_loops)

In [ ]:
result = probe(
    adapter,
    task,
    depths=[2, 4, 6, 8],
    n_samples=128,
    batch_size=64,
)
print(result.as_array())

In [ ]:
from depth_lens.viz import plot_accuracy_curve
from pathlib import Path

plot_accuracy_curve(result, Path('quickstart_curve.png'))
from IPython.display import Image
Image('quickstart_curve.png')

## 4. Diagnostics

In [ ]:
print(f'effective depth (≥0.5 acc): {result.effective_depth(0.5)}')
for d in result.depths:
    over = result.overthinking(d)
    if over:
        print(f'overthinking @ d={d}: peak={over["peak_compute"]} ({over["peak_accuracy"]:.2f}) → last={over["last_compute"]} ({over["last_accuracy"]:.2f})')